In [2]:
pip install -U langchain-openrouter

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.messages import SystemMessage


In [10]:
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)

In [11]:

NOTES = [
    "LangGraph gives an agent memory using a checkpointer, and a thread_id names one conversation. "
    "Same thread_id remembers earlier turns; a new thread_id starts fresh.",

    "A tool is an ordinary Python function with a docstring. The model reads the name, docstring "
    "and typed arguments to decide when and how to call it.",

    "create_agent(model, tools) builds the whole ReAct loop for you: it calls the model, runs the "
    "tool it asks for, feeds the result back, and repeats until done.",

    "RAG (retrieval-augmented generation) means: retrieve relevant text first, then let the model "
    "answer using that text, so answers are grounded in your documents instead of guessed.",

    "Saarathi Academy runs a 12-week AI Engineering and Machine Learning course, two hours a day, "
    "in Old Baneshwor, Kathmandu.",

    "The safe way to run arithmetic from a model is a small ast-based evaluator, never Python's "
    "eval(), because a tool is a door into your system.",
]


In [26]:
import os
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # load from .env - never hardcode
)



In [27]:
me = {"configurable": {"thread_id": "aug_26"}} 

In [28]:
_vec = TfidfVectorizer().fit(NOTES)          # build the index once
_M   = _vec.transform(NOTES)



@tool
def search_notes(query: str) -> str:
    """Search the user's course notes and return the most relevant note."""
    sims = cosine_similarity(_vec.transform([query]), _M)[0]
    return NOTES[int(sims.argmax())]  

@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression, e.g. '9 * 10'."""
    return safe_eval(expression)      # never eval() model input



In [29]:

custom_system_message =SystemMessage(content = """
You are a strict but helpful technical support agent.

Your job is to provide accurate, clear, and easy-to-understand answers.

Follow these rules for every response:

1. STRUCTURE
   - Organize answers logically.
   - Use headings when the response has multiple sections.
   - Use bullet points or numbered lists when appropriate.
   - Separate different ideas into separate paragraphs.

2. CLARITY
   - Use simple, direct language.
   - Avoid unnecessary technical jargon.
   - If a technical term is necessary, briefly explain it.
   - Do not use unnecessarily complicated sentences.

3. CODE
   - When providing code, always use properly formatted code blocks.
   - Explain what the important parts of the code do.
   - If fixing code, clearly identify the problem and then provide the corrected code.

4. ANSWERS
   - Answer the user's question directly first.
   - Do not add irrelevant information.
   - Give explanations step-by-step when the problem requires multiple steps.
   - Highlight important warnings, errors, or key points.

5. READABILITY
   - Keep responses visually clean.
   - Use whitespace between sections.
   - Prefer short paragraphs.
   - Use bullet points instead of large blocks of text whenever possible.

6. TROUBLESHOOTING
   When helping with an error:
   - Identify the error.
   - Explain why it happened.
   - Show the fix.
   - Provide corrected code when appropriate.
   - Mention any additional steps required to prevent the error.

7. TONE
   - Be professional, calm, and helpful.
   - Do not be unnecessarily verbose.
   - Do not repeat the user's question.
   - Never sacrifice accuracy for brevity.

Always prioritize:
Accuracy → Clarity → Structure → Conciseness.

"""
                                    )
assistant = create_agent(system_prompt = custom_system_message, tools=TOOLS ,checkpointer=InMemorySaver(),model=model)
me = {"configurable": {"thread_id": "another_thread_test"}} 


def say(config, text):
    r = assistant.invoke({"messages": [{"role": "user", "content": text}]}, config)
    print("agent:", r["messages"][-1].content)

    




In [30]:
@tool
def decide_for_me(options: list[str]) -> str:
    """Pick one option at random when the user can't decide between several things."""
    if not options:
        return "give me some options first"
    return f"I choose: {random.choice(options)}"
@tool
def roll_dice(sides: int, times: int) -> str:
    """Roll a dice with the given number of sides, a given number of times."""
    rolls = [random.randint(1, sides) for _ in range(times)]
    return f"rolled {rolls}, total = {sum(rolls)}"



TOOLS = [search_notes, calculator, decide_for_me, roll_dice]  

In [31]:
say(me, "find how many weeks of study are in the course and then multiply it by 5 and finally decide for me what to study first")

agent: [{'type': 'text', 'text': '### Course Duration\n\nThe Saarathi Academy AI Engineering and Machine Learning course consists of **12 weeks** of study.\n\n### Calculation\n\nMultiplying the number of study weeks by 5:\n\n* **Calculation:** 12 weeks × 5 = **60**\n\n### Study Recommendation\n\nBased on the topics covered in your course, here is the recommendation on what you should study first:\n\n* **LangGraph Agents**\n\n> **Key Concept:** LangGraph is a framework used to build stateful, multi-actor applications with Large Language Models (LLMs). It allows you to give an agent memory using a checkpointer, where a specific `thread_id` is used to track and remember previous turns in a conversation.', 'extras': {'signature': 'EqIMCp8MARFNMg8fsZ0YtTHYi8ZAFxn6YNi6ZI2mekYENvUmVcYOCdaxeXGDnGlrb/k+OXuAY3GMh/1but/cnHPYTPFIBL2d3rmFTWftz9VCDfso2ImSWqVM74VvDkObVDvvRwTWi1aChjn0eEl3u43fqhNUBVJM6mCvlNoEqOCrh5/3YhaPFYwfo7cW8szXj8KqKsK3EXaSXzhURDk/fhidCXM2i2XQts+oP72O5fXN5xjVIn/opxJkuKYxyaBfjPhBngE